# B.O.S.S. — Test YOLO11 su recordings

**Pipeline:**
1. Ricerca frame (dataset già estratto su Kaggle)
2. Caricamento `best.pt` + inferenza → frame annotati
3. Distribuzione classi e confidenza
4. Preparazione Ground Truth (`BOSS_recordings.yolov8`)
5. Valutazione quantitativa contro GT (mAP, Precision, Recall)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo
8. Report Markdown leggibile da IA

---
**Output — `/kaggle/working/TEST_OUTPUT/` (ricreata da zero a ogni run):**
- `model_output/` — predizioni sui recordings: frame annotati, `predictions.csv`, distribuzione classi/confidenza, frame campione
- `gt_eval/` — grafici valutazione contro Ground Truth (AP per classe, metriche CSV)
- `comparison/` — dashboard di confronto GT vs output modello
- `markdown/` — report `.md` riepilogativi

---
**Kaggle — dati di input (già estratti sotto `/kaggle/input`):**
- **Dataset** recordings: cartelle `frames` / `frames(1)`
- **Dataset** GT: export Roboflow con `train/test/valid` + `data.yaml`
- **Model**: `best.pt`

In [ ]:
import shutil, os
for entry in os.listdir("/kaggle/working"):
    p = os.path.join("/kaggle/working", entry)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("working pulita:", os.listdir("/kaggle/working"))


In [ ]:
# ============================================================
# Cella 1 — Import librerie
# ============================================================
%pip install ultralytics pyyaml tqdm torchmetrics --quiet

import os
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import yaml

import matplotlib.pyplot as plt

from tqdm import tqdm
from ultralytics import YOLO

import torch
from torchmetrics.detection import MeanAveragePrecision

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 1b - Import modulo condiviso
# ============================================================
try:
    import boss_eval_utils as beu  # Kaggle: utility script auto-mounted in sys.path
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/lorenzoverdura/boss_eval_utils")
    import boss_eval_utils as beu
print(f"boss_eval_utils caricato da: {beu.__file__}")

In [ ]:
# ============================================================
# Cella 2 — Configurazione (classi e soglie dal modulo condiviso)
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

# Nome modello REALE (titolo report parametrico - niente hardcode)
MODEL_NAME = "YOLO11"

if IS_KAGGLE:
    # ── Kaggle ──────────────────────────────────────────────
    KAGGLE_DATA_DIR  = Path("/kaggle/input/datasets/lorenzoverdura")
    KAGGLE_MODEL_DIR = Path("/kaggle/input/models/ultralytics/yolo11/pytorch/default/1")

    BASE_DIR       = Path("/kaggle/input")
    MODEL_PATH     = KAGGLE_MODEL_DIR / "yolo11n.pt"
    RECORDINGS_ZIP = KAGGLE_DATA_DIR  / "boss-recordings/images"
    GT_ZIP_PATH    = KAGGLE_DATA_DIR  / "boss-recordings1/BOSS_recordings.yolov11 (2)"
else:
    # ── Locale ──────────────────────────────────────────────
    BASE_DIR       = Path("/home/lorenzoverdura8/BOSS_CODE")
    MODEL_PATH     = BASE_DIR / "best.pt"
    RECORDINGS_ZIP = BASE_DIR / "recordings.zip"
    GT_ZIP_PATH    = BASE_DIR / "BOSS_recordings.yolov8 (2)/rf_split"

OUTPUT_DIR   = Path("/kaggle/working")
GT_EXTRACTED = OUTPUT_DIR / "ground_truth_boss"

TEST_OUTPUT    = OUTPUT_DIR / "TEST_OUTPUT"
DIR_MODEL_OUT  = TEST_OUTPUT / "model_output"
DIR_GT_EVAL    = TEST_OUTPUT / "gt_eval"
DIR_COMPARISON = TEST_OUTPUT / "comparison"
DIR_MARKDOWN   = TEST_OUTPUT / "markdown"

# --- Classi e soglie dal modulo condiviso (no ridondanza) ---
BOSS_CLASSES   = beu.BOSS_CLASSES
NUM_CLASSES    = beu.NUM_CLASSES
CONF_THRESHOLD = beu.CONF_THRESHOLD   # 0.25
IOU_THRESHOLD  = beu.IOU_THRESHOLD    # 0.45 (NMS)
MATCH_IOU      = beu.MATCH_IOU        # 0.50 (TP/FP/FN)

IMG_SIZE       = 640
DEVICE         = "0" if torch.cuda.is_available() else "cpu"

# Caricamento modello YOLO11
trained_model = YOLO(str(MODEL_PATH))
MODEL_CLASSES = trained_model.names
MODEL_NC      = len(MODEL_CLASSES)
MODEL_TO_BOSS = beu.build_model_to_boss(MODEL_CLASSES)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GT_EXTRACTED.mkdir(parents=True, exist_ok=True)

if TEST_OUTPUT.exists():
    shutil.rmtree(TEST_OUTPUT)
for d in (DIR_MODEL_OUT, DIR_GT_EVAL, DIR_COMPARISON, DIR_MARKDOWN):
    d.mkdir(parents=True, exist_ok=True)

mapped_names = sorted({BOSS_CLASSES[b] for b in MODEL_TO_BOSS.values()})
missing      = [c for c in BOSS_CLASSES if c not in mapped_names]
print(f"Ambiente:       {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"MODEL_PATH:     {MODEL_PATH}  —  esiste: {MODEL_PATH.exists()}")
print(f"RECORDINGS_ZIP: {RECORDINGS_ZIP}  —  esiste: {RECORDINGS_ZIP.exists()}")
print(f"GT_ZIP_PATH:    {GT_ZIP_PATH}  —  esiste: {GT_ZIP_PATH.exists()}")
print(f"Classi modello: {MODEL_NC}")
print(f"Classi BOSS coperte ({len(mapped_names)}/{NUM_CLASSES}): {mapped_names}")
print(f"Classi BOSS NON coperte: {missing}")
print(f"CONF_THRESHOLD: {CONF_THRESHOLD}")
print(f"IOU_THRESHOLD:  {IOU_THRESHOLD}")
print(f"DEVICE:         {DEVICE}")

In [ ]:
# ============================================================
# Cella 3 — Ricerca frame (Kaggle: dataset già estratto)
# ============================================================
# Su Kaggle il dataset è già estratto in sola lettura sotto /kaggle/input,
# quindi NON si lavora con gli ZIP. RECORDINGS_ZIP punta direttamente alla
# cartella 'images' (le stesse immagini della GT, senza label). Le immagini
# vengono consolidate in un'unica cartella scrivibile in /kaggle/working (via
# symlink, senza copia), usata come sorgente per l'inferenza (Cella 4).

recordings_root = RECORDINGS_ZIP

# Fallback locale: se RECORDINGS_ZIP è un vero .zip, lo estrae in /kaggle/working.
if recordings_root.is_file() and zipfile.is_zipfile(recordings_root):
    extract_to = OUTPUT_DIR / "recordings"
    if not extract_to.exists() or not any(extract_to.iterdir()):
        print(f"Estrazione {recordings_root} ...")
        with zipfile.ZipFile(recordings_root, "r") as z:
            z.extractall(extract_to)
    recordings_root = extract_to

if not recordings_root.exists():
    raise FileNotFoundError(f"Cartella recordings non trovata: {recordings_root}")

# Cartella scrivibile che consolida i frame trovati sotto recordings_root.
# Symlink (no copia) per risparmiare tempo/disco.
TEST_FRAMES_DIR = OUTPUT_DIR / "frames_all"
if TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
TEST_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Raccoglie ricorsivamente tutte le immagini sotto recordings_root.
img_files = sorted(
    p for p in recordings_root.rglob("*")
    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
if not img_files:
    raise FileNotFoundError(
        f"Nessuna immagine (.jpg/.jpeg/.png) trovata sotto {recordings_root}"
    )

all_frames = []
for img in img_files:
    dst = TEST_FRAMES_DIR / img.name
    try:
        dst.symlink_to(img.resolve())
    except OSError:
        shutil.copy(img, dst)
    all_frames.append(dst)

print(f"Totale frame trovati: {len(all_frames)}")
print(f"TEST_FRAMES_DIR (consolidata): {TEST_FRAMES_DIR}")

In [ ]:
# ============================================================
# Cella 4 — YOLO11 inferenza su tutti i frame → df_preds + frame annotati
# ============================================================
all_predictions  = []
PREDICT_SAVE_DIR = None

inference_results = trained_model.predict(
    source   = str(TEST_FRAMES_DIR),
    conf     = CONF_THRESHOLD,
    iou      = IOU_THRESHOLD,
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    save     = True,
    save_txt = True,
    project  = str(DIR_MODEL_OUT),
    name     = "annotated_frames",
    exist_ok = True,
    stream   = True,
)

for result in inference_results:
    if PREDICT_SAVE_DIR is None:
        PREDICT_SAVE_DIR = Path(result.save_dir)
    frame_name = Path(result.path).name
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue
    cls_ids = boxes.cls.cpu().numpy().astype(int)
    confs   = boxes.conf.cpu().numpy()
    xyxy    = boxes.xyxy.cpu().numpy()
    for cid, cf, box in zip(cls_ids, confs, xyxy):
        boss_id = MODEL_TO_BOSS.get(int(cid))
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   int(cid),
            "class_name": MODEL_CLASSES[int(cid)],
            "boss_class": BOSS_CLASSES[boss_id] if boss_id is not None else None,
            "confidence": float(cf),
            "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3],
        })

df_preds = pd.DataFrame(all_predictions)
print(f"\nCartella frame annotati: {PREDICT_SAVE_DIR}")
print(f"Totale predizioni:       {len(df_preds)}")
print(f"Frame con predizioni:    {df_preds['frame'].nunique() if not df_preds.empty else 0}")
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))
df_preds.to_csv(DIR_MODEL_OUT / "predictions.csv", index=False)
print(f"\nSalvato: {DIR_MODEL_OUT}/predictions.csv")


def run_yolo(img_bgr):
    """
    Inferenza su immagine BGR numpy. Restituisce (boxes_xyxy_abs, class_ids, scores).
    conf=0 per passare tutte le detection a torchmetrics (curva PR completa).
    """
    results = trained_model.predict(
        source=img_bgr, conf=0, iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE, device=DEVICE, verbose=False,
    )
    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        return (np.zeros((0, 4), dtype=np.float32),
                np.array([], dtype=np.int64),
                np.array([], dtype=np.float32))
    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    class_ids  = result.boxes.cls.cpu().numpy().astype(np.int64)
    scores     = result.boxes.conf.cpu().numpy()
    return boxes_xyxy, class_ids, scores

In [ ]:
# ============================================================
# Cella 5 — Distribuzione classi (BOSS) e confidenza
# ============================================================
# Grafico 1: numero rilevamenti per classe BOSS (predizioni mappate).
# Grafico 2: distribuzione confidenza su tutte le predizioni (istogramma).
# Salvati in TEST_OUTPUT/model_output/.

if df_preds.empty:
    print("Nessuna predizione — grafici saltati.")
else:
    # Grafico 1: rilevamenti per classe BOSS — ordine fisso = BOSS_CLASSES
    df_boss = df_preds.dropna(subset=["boss_class"])
    class_counts = (
        df_boss["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = plt.cm.tab20.colors[:NUM_CLASSES]
    ax.bar(class_counts.index, class_counts.values, color=colors)
    ax.set_title("Distribuzione classi BOSS rilevate — recordings", fontsize=14)
    ax.set_xlabel("Classe BOSS")
    ax.set_ylabel("Numero rilevamenti")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_class_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

    # Grafico 2: distribuzione confidenza (tutte le predizioni)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(
        CONF_THRESHOLD, color="red", linestyle="--",
        label=f"soglia={CONF_THRESHOLD} (allineata al training)"
    )
    ax.set_title("Distribuzione confidenza predizioni", fontsize=13)
    ax.set_xlabel("Confidenza")
    ax.set_ylabel("Frequenza")
    ax.legend()
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_confidence_distribution.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 6 — Preparazione Ground Truth (remap → spazio classi modello)
# ============================================================
# La GT (export Roboflow, già estratta in sola lettura) ha le sue classi in
# data.yaml. Per valutare QUALSIASI modello con val() NATIVO (nessun mismatch
# di nc), le label GT vengono riportate nello spazio ID del modello tramite la
# mappa universale:   gt_id → nome → classe BOSS → model_id
# Le classi BOSS non presenti nel modello vengono scartate dalla GT (il modello
# non può rilevarle → escluse dalla valutazione, niente metriche sfalsate).

gt_root = GT_ZIP_PATH

# Pulisce la prep precedente (ID space diverso tra un modello e l'altro)
gt_test_dir = GT_EXTRACTED / "test"
if gt_test_dir.exists():
    shutil.rmtree(gt_test_dir)

# Legge il data.yaml della GT per ottenere l'ordine classi originale
gt_yaml_in_path = gt_root / "data.yaml"
if not gt_yaml_in_path.exists():
    raise FileNotFoundError(f"data.yaml non trovato in: {gt_yaml_in_path}")
with open(gt_yaml_in_path) as f:
    gt_yaml_in = yaml.safe_load(f)
GT_CLASSES = gt_yaml_in["names"]
print(f"Classi GT ({len(GT_CLASSES)}): {GT_CLASSES}")

# Inverso della mappa modello→BOSS: boss_id → model_id (prima occorrenza)
BOSS_TO_MODEL = {}
for mid, bid in MODEL_TO_BOSS.items():
    BOSS_TO_MODEL.setdefault(bid, mid)

# gt_id → model_id, passando per la classe BOSS canonica (mappa universale)
GT_TO_MODEL = {}
unmapped = []
for gt_id, gt_name in enumerate(GT_CLASSES):
    boss = beu.resolve_to_boss(gt_name)
    if boss is None:
        unmapped.append(gt_name)
        continue
    model_id = BOSS_TO_MODEL.get(BOSS_CLASSES.index(boss))
    if model_id is None:
        unmapped.append(gt_name)
        continue
    GT_TO_MODEL[gt_id] = model_id
print(f"Mappa GT→modello: {GT_TO_MODEL}")
print(f"Classi GT scartate (non rilevabili dal modello): {unmapped}")


def remap_labels(src_label_dir, dst_label_dir, id_map):
    """Copia i file .txt YOLO rimappando i class ID secondo id_map (scarta i non mappati)."""
    src = Path(src_label_dir)
    dst = Path(dst_label_dir)
    dst.mkdir(parents=True, exist_ok=True)
    remapped, dropped = 0, 0
    for lf in src.glob("*.txt"):
        out_lines = []
        for line in lf.read_text().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            gid = int(parts[0])
            if gid not in id_map:
                dropped += 1
                continue
            parts[0] = str(id_map[gid])
            out_lines.append(" ".join(parts))
            remapped += 1
        (dst / lf.name).write_text("\n".join(out_lines))
    print(f"  Annotazioni rimappate: {remapped} | Righe scartate: {dropped}")


# Usa SOLO lo split 'test' (contiene images/ e labels/).
# I nomi file restano invariati così ogni immagine combacia con la sua label.
GT_TEST_IMGS   = GT_EXTRACTED / "test" / "images"
GT_TEST_LABELS = GT_EXTRACTED / "test" / "labels"
GT_TEST_IMGS.mkdir(parents=True, exist_ok=True)

src_imgs   = gt_root / "test" / "images"
src_labels = gt_root / "test" / "labels"
if not src_imgs.exists():
    raise FileNotFoundError(f"Cartella immagini test non trovata: {src_imgs}")

total_imgs = 0
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for img in src_imgs.glob(ext):
        dst = GT_TEST_IMGS / img.name
        try:
            dst.symlink_to(img.resolve())
        except OSError:
            shutil.copy(img, dst)
        total_imgs += 1
print(f"  Split 'test': {total_imgs} immagini")

if not src_labels.exists():
    raise FileNotFoundError(f"Cartella label test non trovata: {src_labels}")
remap_labels(src_labels, GT_TEST_LABELS, GT_TO_MODEL)

print(f"\nTotale immagini GT (test/): {total_imgs}")

# data.yaml nello spazio classi del MODELLO (val() nativo, nc allineato)
model_names_list = [MODEL_CLASSES[i] for i in range(MODEL_NC)]
gt_data_yaml = {
    "path":  str(GT_EXTRACTED),
    "train": "",
    "val":   "test/images",
    "test":  "",
    "nc":    MODEL_NC,
    "names": model_names_list,
}
gt_yaml_out = GT_EXTRACTED / "data_boss.yaml"
with open(gt_yaml_out, "w") as f:
    yaml.dump(gt_data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f"data_boss.yaml scritto in: {gt_yaml_out}")

In [ ]:
# ============================================================
# Cella 7 — Valutazione quantitativa (torchmetrics + TP/FP/FN greedy matching)
# ============================================================
# Calcola mAP@0.5, mAP@0.5:0.95 con torchmetrics. Precision e Recall reali
# via greedy matching IoU (TP/FP/FN) dal modulo condiviso, a IoU=MATCH_IOU e
# soglia CONF_THRESHOLD. Stesso metodo degli altri notebook.

metric_50   = MeanAveragePrecision(iou_type="bbox", class_metrics=True, iou_thresholds=[0.5])
metric_full = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

# Accumulatori TP/FP/FN per classe (model class_id)
tp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fp_pc = np.zeros(MODEL_NC, dtype=np.int64)
fn_pc = np.zeros(MODEL_NC, dtype=np.int64)

gt_label_files = sorted(GT_TEST_LABELS.glob("*.txt"))
processed = 0

for label_path in tqdm(gt_label_files, desc="Valutazione GT (YOLO11)"):
    img_name = label_path.stem
    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        p = GT_TEST_IMGS / (img_name + ext)
        if p.exists():
            img_path = p
            break
    if img_path is None:
        continue

    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    h, w = img_bgr.shape[:2]

    # Parse GT YOLO — gestisce sia bbox 'cls cx cy w h' sia poligoni di
    # segmentazione 'cls x1 y1 x2 y2 ...' (l'export Roboflow yolo11-seg salva
    # poligoni): per i poligoni il bbox è ricavato da min/max dei vertici.
    gt_boxes, gt_labels = [], []
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        sid = int(parts[0])
        coords = [float(v) for v in parts[1:]]
        if len(coords) == 4:
            cx, cy, bw, bh = coords
            x1 = (cx - bw/2) * w;  y1 = (cy - bh/2) * h
            x2 = (cx + bw/2) * w;  y2 = (cy + bh/2) * h
        else:
            xs = coords[0::2];  ys = coords[1::2]
            x1 = min(xs) * w;  y1 = min(ys) * h
            x2 = max(xs) * w;  y2 = max(ys) * h
        gt_boxes.append([x1, y1, x2, y2])
        gt_labels.append(sid)

    if not gt_boxes:
        continue

    targets = [{"boxes":  torch.tensor(gt_boxes,  dtype=torch.float32),
                "labels": torch.tensor(gt_labels, dtype=torch.int64)}]

    # Inferenza senza filtro conf: torchmetrics costruisce la curva PR completa
    boxes_xyxy, class_ids, scores = run_yolo(img_bgr)
    preds = [{"boxes":  torch.tensor(boxes_xyxy, dtype=torch.float32),
              "scores": torch.tensor(scores,     dtype=torch.float32),
              "labels": torch.tensor(class_ids,  dtype=torch.int64)}]

    metric_50.update(preds, targets)
    metric_full.update(preds, targets)

    # TP/FP/FN per classe: greedy matching @ conf CONF_THRESHOLD, IoU MATCH_IOU
    keep = scores >= CONF_THRESHOLD
    beu.accumulate_matches(
        tp_pc, fp_pc, fn_pc,
        boxes_xyxy[keep], class_ids[keep], scores[keep],
        gt_boxes, gt_labels, MATCH_IOU,
    )

    processed += 1

print(f"Immagini valutate: {processed}")

res_50   = metric_50.compute()
res_full = metric_full.compute()

map50   = float(res_50["map"])
map5095 = float(res_full["map"])

# Filtra classi con AP=-1 (nessun match GT o pred)
classes_t = res_50["classes"]
ap50_t    = res_50["map_per_class"]
ap5095_t  = res_full["map_per_class"]

valid_cls           = (ap50_t >= 0) & (ap5095_t >= 0)
gt_class_indices    = classes_t[valid_cls].numpy().astype(int)
gt_per_class_ap50   = ap50_t[valid_cls].numpy()
gt_per_class_ap5095 = ap5095_t[valid_cls].numpy()

# Precision/Recall reali (TP/FP/FN) per le sole classi valutate da torchmetrics
prec_pc, rec_pc = beu.precision_recall_per_class(tp_pc, fp_pc, fn_pc)
gt_per_class_p = prec_pc[gt_class_indices]
gt_per_class_r = rec_pc[gt_class_indices]

precision = float(gt_per_class_p.mean()) if len(gt_per_class_p) > 0 else 0.0
recall    = float(gt_per_class_r.mean()) if len(gt_per_class_r) > 0 else 0.0
f1_score  = 2 * precision * recall / (precision + recall + 1e-9)

GT_EVAL_SAVE_DIR = DIR_GT_EVAL

print(f"\nmAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"Recall:        {recall:.4f}  (TP/FP/FN, IoU {MATCH_IOU}, conf {CONF_THRESHOLD})")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Classi valutate: {len(gt_class_indices)}")

In [ ]:
# ============================================================
# Cella 8 — Metriche per classe + grafici GT (YOLO11)
# ============================================================
def _boss_label(model_id):
    bid = MODEL_TO_BOSS.get(int(model_id))
    return BOSS_CLASSES[bid] if bid is not None else MODEL_CLASSES.get(int(model_id), str(model_id))

gt_class_names  = [_boss_label(i) for i in gt_class_indices]
gt_per_class_f1 = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

df_metrics = pd.DataFrame({
    "Classe":      gt_class_names,
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
df_display[float_cols] = df_display[float_cols].applymap(lambda x: f"{x:.4f}")
print("=== Metriche per Classe (YOLO11) ===")
print(df_display.to_string(index=False))
df_metrics.to_csv(DIR_GT_EVAL / "metrics_per_class_gt.csv", index=False)
print(f"\nSalvato: {DIR_GT_EVAL}/metrics_per_class_gt.csv")

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

# Grafico 1: AP@0.5 e AP@0.5:0.95 per classe
x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title("AP per classe — Ground Truth (YOLO11)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_ap_per_class_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

# Grafico 2: scatter Precision vs Recall per classe
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in df_plot.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per classe — Ground Truth (YOLO11)")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = DIR_GT_EVAL / "plot_pr_scatter_gt.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 9 — Griglia campione frame annotati
# ============================================================
# Mostra N_SAMPLES frame con bounding box disegnate da YOLOv8,
# letti da PREDICT_SAVE_DIR (cartella output dell'inferenza).
# Salvata in TEST_OUTPUT/model_output/.

N_SAMPLES = 9

if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = (
        list(PREDICT_SAVE_DIR.glob("*.jpg")) +
        list(PREDICT_SAVE_DIR.glob("*.png"))
    )

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings — predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    save_path = DIR_MODEL_OUT / "plot_sample_predictions.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 10 - Efficienza: GMACs / GFLOPs / parametri su grafo ONNX
# ============================================================
# Contatore uniforme (onnx-tool, modulo condiviso beu) per tutti i notebook.
# YOLO11 esportato in ONNX alla STESSA risoluzione di inferenza (IMG_SIZE),
# shape fissa. GMACs = MACs/1e9 ; GFLOPs = 2*GMACs. Eseguito PRIMA di dashboard
# e report cosi i valori di efficienza compaiono in entrambi.
%pip install onnx-tool --quiet

DIR_EFFICIENCY = TEST_OUTPUT / "efficiency"
DIR_EFFICIENCY.mkdir(parents=True, exist_ok=True)

# Esporta in ONNX a shape fissa, risoluzione IMG_SIZE. L'export scrive l'ONNX
# accanto al .pt sorgente: i pesi vengono copiati in una cartella scrivibile
# perche /kaggle/input e read-only.
eff = None
df_eff = None
try:
    export_pt = DIR_EFFICIENCY / Path(str(MODEL_PATH)).name
    shutil.copy(MODEL_PATH, export_pt)
    export_model = YOLO(str(export_pt))
    ONNX_PATH = Path(export_model.export(format="onnx", imgsz=IMG_SIZE, opset=13, dynamic=False, simplify=False))
    # ONNX_PATH = Path("/kaggle/input/PLACEHOLDER-yolo-onnx/yolo11n.onnx")   # <<< (opzionale) ONNX pronto

    eff = beu.count_onnx_flops_params(ONNX_PATH)
    df_eff = pd.DataFrame([{
        "model":    MODEL_NAME,
        "onnx":     ONNX_PATH.name,
        "input_h":  IMG_SIZE,
        "input_w":  IMG_SIZE,
        "params":   eff["params"],
        "params_M": round(eff["params"] / 1e6, 4),
        "gmacs":    round(eff["gmacs"], 4),
        "gflops":   round(eff["gflops"], 4),
    }])
    eff_csv = DIR_EFFICIENCY / "flops_params.csv"
    df_eff.to_csv(eff_csv, index=False)
    print(df_eff.to_string(index=False))
    print(f"\nSalvato: {eff_csv}")
except Exception as e:
    print(f"[efficienza] calcolo GMACs non riuscito: {e}")
    print("Dashboard e report proseguiranno con GMACs = n/d.")

In [ ]:
# ============================================================
# Cella 11 — Dashboard riepilogo finale (confronto GT vs modello)
# ============================================================
# Dashboard 6-pannelli che combina metriche GT aggregate, AP per classe,
# F1 per classe, scatter P/R, distribuzione classi rilevate e configurazione.
# Salvata in TEST_OUTPUT/comparison/.

fig = plt.figure(figsize=(16, 10))
fig.suptitle("B.O.S.S. — Riepilogo Test su recordings", fontsize=16, fontweight="bold")

# Pannello 1: metriche aggregate GT come barre orizzontali
ax1 = fig.add_subplot(2, 3, 1)
metrics_summary = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()), color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

# Pannello 2: AP@0.5 per classe
ax2 = fig.add_subplot(2, 3, 2)
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

# Pannello 3: F1 per classe
ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

# Pannello 4: scatter Precision vs Recall per classe
ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i % len(scatter_colors)], s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

# Pannello 5: distribuzione classi BOSS rilevate sui frame recordings
ax5 = fig.add_subplot(2, 3, 5)
df_boss5 = df_preds.dropna(subset=["boss_class"]) if not df_preds.empty else df_preds
if not df_preds.empty and not df_boss5.empty:
    class_counts_test = df_boss5["boss_class"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 7})
    ax5.set_title("Distribuzione classi BOSS — Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione mappata", ha="center", va="center")
    ax5.axis("off")

# Pannello 6: info configurazione
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
frames_count = len(all_frames)
pred_count   = len(df_preds)
pred_mapped  = int(df_preds["boss_class"].notna().sum()) if not df_preds.empty else 0
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
_eff       = globals().get("eff")
gmacs_str  = f"{_eff['gmacs']:.2f}" if _eff else "n/d"
params_str = f"{_eff['params']/1e6:.2f} M" if _eff else "n/d"
info_text = (
    f"Modello:        {Path(str(MODEL_PATH)).name}\n"
    f"Classi modello: {MODEL_NC}\n"
    f"Classi BOSS:    {NUM_CLASSES}\n"
    f"Immagine:       {IMG_SIZE}x{IMG_SIZE}\n"
    f"Conf threshold: {CONF_THRESHOLD}  (= training)\n"
    f"IoU threshold:  {IOU_THRESHOLD}\n"
    f"Device:         {DEVICE}\n"
    f"GMACs (ONNX):   {gmacs_str}\n"
    f"Parametri:      {params_str}\n\n"
    f"Frame recordings:  {frames_count}\n"
    f"Pred. totali:      {pred_count}\n"
    f"Pred. mappate BOSS:{pred_mapped}\n"
    f"GT immagini test:  {gt_img_count}\n"
    f"Output:            TEST_OUTPUT/\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
save_path = DIR_COMPARISON / "plot_dashboard_riepilogo.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato: {save_path}")

In [ ]:
# ============================================================
# Cella 12 — Report Markdown (modulo condiviso, titolo parametrico)
# ============================================================
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))

if not df_preds.empty:
    class_dist = (
        df_preds.dropna(subset=["boss_class"])["boss_class"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    pred_total       = len(df_preds)
    frames_with_pred = df_preds["frame"].nunique()
else:
    class_dist = None
    pred_total = 0
    frames_with_pred = 0

_eff = globals().get("eff")
config = {
    "Modello":                     Path(str(MODEL_PATH)).name,
    "Classi modello":              MODEL_NC,
    "Classi BOSS":                 NUM_CLASSES,
    "Dimensione immagine":         f"{IMG_SIZE}x{IMG_SIZE}",
    "Conf threshold (inferenza)":  CONF_THRESHOLD,
    "IoU threshold (NMS)":         IOU_THRESHOLD,
    "Match IoU (TP/FP/FN)":        MATCH_IOU,
    "Device":                      DEVICE,
    "GMACs (ONNX)":                (round(_eff["gmacs"], 3) if _eff else "n/d"),
    "Parametri (M)":               (round(_eff["params"] / 1e6, 3) if _eff else "n/d"),
    "Frame recordings":            len(all_frames),
    "Immagini GT (test)":          gt_img_count,
}
metrics = {
    "map50": map50, "map5095": map5095,
    "precision": precision, "recall": recall, "f1": f1_score,
    "match_iou": MATCH_IOU, "conf_threshold": CONF_THRESHOLD,
}

report = beu.build_report(
    MODEL_NAME, config, metrics, df_metrics,
    class_distribution=class_dist, pred_total=pred_total,
    frames_with_pred=frames_with_pred, n_frames=len(all_frames),
)

report_path = DIR_MARKDOWN / "report.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report Markdown scritto: {report_path}")

metrics_md_path = DIR_MARKDOWN / "metrics_per_class.md"
metrics_md_path.write_text(
    f"# Metriche per classe - Ground Truth ({MODEL_NAME})\n\n" + beu.df_to_md(df_metrics) + "\n",
    encoding="utf-8",
)
print(f"Tabella metriche scritta: {metrics_md_path}")